# 13 · Use Case — Energy / Electricity Load Forecasting

Hourly electricity load has strong **daily and weekly** seasonality. TimesFM's
long context (up to 16k points) captures both. Useful for utilities, building
managers, and energy traders.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
# 30 days of hourly load (720 points) with daily + weekly cycles
rng = np.random.default_rng(50)
h = np.arange(24*30)
load = (
    3000
    + 800*np.sin(2*np.pi*(h % 24)/24 - 1.0)          # daily peak in evening
    + 300*np.sin(2*np.pi*h/(24*7))                    # weekly cycle
    + rng.normal(0, 80, h.size)
).astype(np.float32)
print("hours of history:", load.size)

In [ ]:
horizon = 48  # forecast 2 days ahead, hourly
point, q = model.forecast(horizon=horizon, inputs=[load])
point, q = point[0], q[0]

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
hist = load[-24*5:]                       # last 5 days
xf = range(len(hist), len(hist)+horizon)
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(range(len(hist)), hist, color="tab:blue", lw=1, label="load (5d)")
ax.plot(xf, point, color="tab:red", lw=1.5, label="48h forecast")
ax.fill_between(xf, q[:,1], q[:,9], color="tab:red", alpha=0.2, label="80% band")
ax.set_title("Hourly electricity load — 48h ahead")
ax.set_xlabel("hour"); ax.set_ylabel("MW"); ax.legend()
fig.tight_layout(); fig.savefig("energy_load.png", dpi=130)
print("saved energy_load.png")

In [ ]:
# Peak-hour analysis for the next 2 days
peak_h = int(np.argmax(point))
print(f"predicted peak in +{peak_h}h at {point[peak_h]:.0f} MW "
      f"(80% band {q[peak_h,1]:.0f}-{q[peak_h,9]:.0f} MW)")

### Where the money is
- **Utilities:** buy energy ahead at the right volume → avoid imbalance penalties.
- **Building managers:** pre-cool/shift load away from predicted peaks.
- **Traders:** position around forecast peaks vs. market prices.